# LAB-D4-02: Model Detective - Curves and Error Buckets

**Purpose:** Diagnose unlabeled learning evidence, then turn aligned prediction failures into a prioritized and falsifiable next experiment.

**Objectives:** `OBJ-D4-02`, `OBJ-D4-03`  
**Estimated duration:** 40 minutes live; core CPU analysis under 30 seconds  
**Prerequisites:** [LESSON-D4-02](../student-guide/day-4-student-guide.md#lesson-d4-02---read-curves-as-competing-hypotheses), [LESSON-D4-03](../student-guide/day-4-student-guide.md#lesson-d4-03---error-analysis-turns-an-average-into-a-work-queue), [ACT-D4-02](../challenges/day-4-challenges.md#act-d4-02---mystery-curves), and Day 3 curve evidence  
**Environment:** CPU; NumPy, matplotlib, scikit-learn; packaged fixed artifacts; no network or training required

Workflow: **Observe anonymous curves -> Predict -> Cite two observations -> Inspect aggregate -> Drill into errors -> Define slices -> Quantify prevalence/severity/payoff -> Propose one disconfirming experiment**.

In [ ]:
import hashlib
import json
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.metrics import confusion_matrix

started = time.perf_counter()
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.2})

def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "courseware/shared/data/day-4/manifest.json").exists(): return candidate
    raise FileNotFoundError("Run from the repository checkout with courseware/shared/data/day-4 present.")

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()

ROOT = find_repo_root(); DATA_DIR = ROOT / "courseware/shared/data/day-4"
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(f"Python {platform.python_version()} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")

## Local Artifact Contract

The four curve bundles are deterministic and unlabeled. The digits artifact contains aligned validation sample IDs, images, labels, probabilities, predictions, confidence, and label-blind image metadata. It contains no model configuration, diagnosis, bucket answer, or test evidence.

In [ ]:
for key in ["mystery_curves", "digits_evidence"]:
    record = manifest["artifacts"][key]
    path = DATA_DIR / record["path"]
    assert path.is_file() and file_sha256(path) == record["sha256"]
curves = np.load(DATA_DIR / manifest["artifacts"]["mystery_curves"]["path"], allow_pickle=False)
evidence = np.load(DATA_DIR / manifest["artifacts"]["digits_evidence"]["path"], allow_pickle=False)
print("Verified artifact checksums and aligned arrays.")

## Observe Anonymous Curves

All bundles use the same neutral colors and axes. Configuration names are intentionally absent. Diagnose from relationships, not styling.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for bundle_index, (bundle_id, ax) in enumerate(zip(curves["bundle_ids"], axes.ravel())):
    ax.plot(curves["epochs"], curves["train_loss"][bundle_index], color="#3b6c8e", label="train loss")
    ax.plot(curves["epochs"], curves["val_loss"][bundle_index], color="#9b5d42", label="validation loss")
    ax.set(title=f"Mystery {bundle_id}", xlabel="epoch", ylabel="loss"); ax.legend()
plt.tight_layout(); plt.show()

## Predict Before Labels

For each bundle, record a leading pattern, two curve observations, one competing explanation, and one evidence request. Use the vocabulary healthy fit, limited fit, widening generalization gap, and unstable/stalled optimization only as hypotheses, not unique causes.

In [ ]:
curve_diagnoses = {
    bundle: {"leading_pattern": "", "observation_one": "", "observation_two": "", "alternative": "", "requested_evidence": ""}
    for bundle in curves["bundle_ids"].tolist()
}
assert all(all(value.strip() for value in response.values()) for response in curve_diagnoses.values())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for bundle_index, (bundle_id, ax) in enumerate(zip(curves["bundle_ids"], axes.ravel())):
    ax.plot(curves["epochs"], curves["train_accuracy"][bundle_index], color="#3b6c8e", label="train")
    ax.plot(curves["epochs"], curves["val_accuracy"][bundle_index], color="#9b5d42", label="validation")
    ax.set(title=f"Requested accuracy evidence: {bundle_id}", xlabel="epoch", ylabel="accuracy", ylim=(0,1.02)); ax.legend()
plt.tight_layout(); plt.show()

## Aggregate Digits Baseline

Treat each 8x8 image as a routing-code glyph. First verify alignment, then inspect accuracy and the complete `(10,10)` confusion matrix.

In [ ]:
sample_ids = evidence["sample_ids"]
images, labels = evidence["images"], evidence["labels"]
probabilities, predictions, confidence = evidence["probabilities"], evidence["predictions"], evidence["confidence"]
assert images.shape[0] == labels.shape[0] == probabilities.shape[0] == len(sample_ids)
assert probabilities.shape[1] == 10 and np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)
accuracy = float(np.mean(predictions == labels))
confusion = confusion_matrix(labels, predictions, labels=np.arange(10))
assert 0.90 <= accuracy <= 0.96 and confusion.shape == (10, 10)
print(f"Fixed validation accuracy: {accuracy:.3f}; errors: {np.sum(predictions != labels)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(confusion, cmap="Greys")
for (row, col), value in np.ndenumerate(confusion):
    if value: ax.text(col, row, str(value), ha="center", va="center", fontsize=8)
ax.set(title="Routing-code confusion matrix", xlabel="predicted", ylabel="actual", xticks=range(10), yticks=range(10)); plt.show()
pair_counts = confusion.copy(); np.fill_diagonal(pair_counts, 0)
top_pairs = np.dstack(np.unravel_index(np.argsort(pair_counts.ravel())[::-1][:5], pair_counts.shape))[0]
print("Top directed confusion pairs:", [(int(a), int(b), int(pair_counts[a,b])) for a,b in top_pairs])

## Predict Before the Error Gallery

Choose two likely high-value error categories from confusion counts alone. Then state why selected images cannot substitute for prevalence counts.

In [ ]:
gallery_predictions = {"two_candidate_categories": "", "likely_high_cost_pair": "", "why_gallery_is_not_prevalence": ""}
assert all(value.strip() for value in gallery_predictions.values())

In [ ]:
error_indices = np.flatnonzero(predictions != labels)
ranked_errors = error_indices[np.argsort(confidence[error_indices])[::-1]]
gallery_indices = ranked_errors[:12]
fig, axes = plt.subplots(3, 4, figsize=(9, 7))
for index, ax in zip(gallery_indices, axes.ravel()):
    ax.imshow(images[index], cmap="gray_r")
    ax.set_title(f"id {sample_ids[index]} | y={labels[index]} p={predictions[index]}\nscore={confidence[index]:.2f}", fontsize=9)
    ax.axis("off")
plt.suptitle("Highest-confidence errors; examples are evidence, not prevalence")
plt.tight_layout(); plt.show()
assert len(gallery_indices) > 0

## Focused TODO: Two Reproducible Slices

Define at least two boolean masks from supplied metadata. Suggested starting points are high border ink and low total ink; thresholds must be computed from the fixed artifact, not chosen after reading labels. Masks may overlap, but report overlap explicitly.

In [ ]:
slice_masks = {
    "high_border_ink": None,  # TODO: boolean mask, for example from a fixed metadata quantile.
    "low_total_ink": None,    # TODO: boolean mask with the same length.
}
assert all(isinstance(mask, np.ndarray) and mask.dtype == bool and mask.shape == labels.shape for mask in slice_masks.values())

In [ ]:
def slice_row(name, mask, severity_weight):
    count = int(mask.sum()); errors = int(np.sum(mask & (predictions != labels)))
    return {
        "slice": name, "count": count, "prevalence": count / len(labels), "errors": errors,
        "error_rate": errors / max(count, 1), "severity_weight": severity_weight,
        "payoff_proxy": errors * severity_weight,
    }

severity_weights = {"high_border_ink": 2.0, "low_total_ink": 1.0}
slice_table = [slice_row(name, mask, severity_weights[name]) for name, mask in slice_masks.items()]
overlap = int(np.logical_and.reduce(list(slice_masks.values())).sum())
for row in slice_table: print(row)
print("Slice overlap:", overlap)
assert all(row["count"] > 0 for row in slice_table)

## Build Error Buckets Without Double Counting

Assign every error to one primary bucket using a declared precedence. The starter precedence is: high-confidence -> high-border -> low-ink -> other. You may rename categories after inspecting images, but do not encode a diagnosis in the artifact itself.

In [ ]:
primary_bucket = np.full(len(labels), "not_error", dtype=object)
remaining = predictions != labels
for name, mask in [
    ("high_confidence", confidence >= 0.60),
    ("high_border_ink", slice_masks["high_border_ink"]),
    ("low_total_ink", slice_masks["low_total_ink"]),
]:
    assigned = remaining & mask
    primary_bucket[assigned] = name
    remaining &= ~assigned
primary_bucket[remaining] = "other_error"
bucket_names, bucket_counts = np.unique(primary_bucket[predictions != labels], return_counts=True)
assert bucket_counts.sum() == len(error_indices)
print("Exclusive error buckets:", dict(zip(bucket_names, bucket_counts)))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
payoff_names = [row["slice"] for row in slice_table]
payoff_values = [row["payoff_proxy"] for row in slice_table]
ax.bar(payoff_names, payoff_values, color=["#3b6c8e", "#9b5d42"])
ax.set(title="Payoff proxy combines observed errors and declared severity", ylabel="errors x severity weight")
plt.show()

## Prioritize One Next Experiment

Frequency, severity, fixability, and spillover can disagree. State one experiment and the evidence that would disconfirm its mechanism. Do not propose several simultaneous changes.

In [ ]:
next_experiment = {
    "priority_bucket": "", "prevalence_evidence": "", "severity_assumption": "",
    "single_intervention": "", "predicted_observation": "", "disconfirming_evidence": "",
    "competing_explanation": "", "stop_rule": "",
}
assert all(value.strip() for value in next_experiment.values())

## Deliberate Failure and Recovery

Failure: choose the largest confusion pair and prescribe a bigger model without inspecting train/validation evidence, images, metadata, or labels. Recovery: connect one quantified bucket to one intervention and one falsifying result.

In [ ]:
failed_draft = {"rule": "largest pair -> bigger model", "attribution_supported": False}
assert not failed_draft["attribution_supported"]
print("Expected design failure caught: aggregate frequency alone does not identify cause or remedy.")

## Optional Challenge: Add One Non-Overlapping Bucket

Define one additional bucket from confidence or image metadata, insert it into the precedence, and verify that all errors are still counted exactly once. Explain what this category makes visible and what it cannot establish causally.

In [ ]:
RUN_OPTIONAL_BUCKET = False
optional_bucket = {"name": "", "definition": "", "interpretation_limit": ""}
if RUN_OPTIONAL_BUCKET:
    assert all(value.strip() for value in optional_bucket.values())
    optional_mask = (confidence < 0.45) & (predictions != labels)
    print("Optional low-confidence errors:", int(optional_mask.sum()))

## Reflection, Takeaways, and Troubleshooting

- A curve pattern supports hypotheses; it does not uniquely name a cause.
- Aggregate -> confusion pair -> examples -> reproducible slices is a narrowing evidence path.
- Report prevalence and consequence before proposing a fix.
- A next experiment earns value by separating plausible explanations.

| Symptom | Likely cause | Recovery |
|---|---|---|
| Checksum mismatch | Stale or modified artifact | Regenerate with the packaged script |
| Images/labels disagree | Arrays were reordered independently | Reload the aligned artifact |
| Bucket totals exceed errors | Overlap was not resolved | Declare precedence or use multi-label accounting explicitly |
| Core exceeds 30 seconds | Training was added | Restore fixed-artifact analysis |

In [ ]:
assert 0.90 <= accuracy <= 0.96 and confusion.shape == (10, 10)
assert len(slice_table) >= 2 and bucket_counts.sum() == len(error_indices)
assert all(value.strip() for value in next_experiment.values())
print(f"LAB-D4-02 checkpoint passed in {time.perf_counter() - started:.2f}s: curves diagnosed, errors accounted, slices quantified, and one falsifiable experiment prioritized.")

## Continue

Use the Day 4 guide debrief: [LAB-D4-02 Debrief - From Symptom to Priority](../student-guide/day-4-student-guide.md#lab-d4-02-debrief---from-symptom-to-priority).